# Frozen knowledge, and our first agent

<img src="images/frozen_intro_vignette.jpg" width="150" alt="Frozen knowledge" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Part 4 ended with a warning: once training stops, a model's weights are frozen — what it "knows" is exactly what was in its training data at that moment. For anything that happens after that training cutoff, a raw model can only guess, hallucinate, or admit it doesn't know. In this part we open up the final black box: how we can break this barrier, connect our model to external tools, and build our first autonomous **agent** that decides for itself how to get the facts.

## Frozen at training time

<img src="images/nobel_prize_vignette.jpg" width="150" alt="Frozen at training time" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Let's ask a question whose answer changes over time: who won the most recent edition of an annual prize. Training data has a cutoff date, so a model cannot know about events that happened after that date. We'll ask about the Nobel Prize in Literature.

In [ ]:
# Program 1: asking a current-events question

import textwrap

from agents import Agent, Runner, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
ollama_model = OpenAIChatCompletionsModel(model="mistral:latest", openai_client=ollama_client)

# A plain agent, no tools yet -- we'll reuse this same one in the next few programs,
# and give it a tool later on, instead of switching to a different API mid-notebook.
research_agent = Agent(
    name="Research Agent",
    instructions="Answer the user's question as best you can.",
    model=ollama_model,
)

def show(text, width=100):
    """Print text wrapped at `width` columns, instead of one long, hard-to-scroll line."""
    print(textwrap.fill(text, width=width))

question = "Who won the most recent Nobel Prize in Literature?"
result = await Runner.run(research_agent, question)
show(result.final_output)

Run it a few times, and pay attention to *what kind* of answer you get: it might confidently name a laureate from several years ago (with no way to tell you that's not the most recent one), or it might hedge and mention it can't be sure past a certain date. Either way, this is exactly the limitation from Part 4, now observed directly: the model isn't looking anything up, it's reproducing whatever pattern its frozen weights encode.

## Asking the model how it would find out

<img src="images/search_strategy_vignette.jpg" width="150" alt="Strategy consulting" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Rather than trying to fix this ourselves right away, let's ask the model itself for a strategy.

In [ ]:
# Program 2: asking the model how it would verify its own answer

question = (
    "You have no live internet access and your training data has a cutoff date. In one or two "
    "sentences, concretely, how would someone find out who won the most recent Nobel Prize in "
    "Literature?"
)
result = await Runner.run(research_agent, question)
show(result.final_output)

Typically, the model suggests something like "check the Nobel Prize's official website" or "search recent news" — sensible advice, it just can't act on it itself. So let's act on it *for* it.

## Fetching real information ourselves

<img src="images/wikipedia_fetch_vignette.jpg" width="150" alt="Fetching information" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Wikipedia exposes a free, public search API over plain HTTP — no API key needed. We'll query it directly with our python code to get the official Nobel pages.

In [ ]:
# Program 3: fetching real, current text ourselves (no API key needed)

import requests
import re

# Wikipedia's API rejects requests with no identifying User-Agent -- see the robot policy above.
HEADERS = {
    "User-Agent": "PLIDOagent-course/1.0 (educational use; contact: laurent.toutain@gmail.com)"
}

def wikipedia_search_snippets(query, limit=3):
    """Search Wikipedia and return a few short text snippets mentioning `query`."""
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query", "list": "search", "srsearch": query, "format": "json", "srlimit": limit
    }
    response = requests.get(url, params=params, headers=HEADERS)
    results = response.json()["query"]["search"]

    # Wikipedia wraps the matched words in <span> tags inside the snippet; strip them out.
    return [f"{r['title']}: {re.sub(r'<[^>]+>', '', r['snippet'])}" for r in results]

# Note the "2025" baked into this query: even this "automatic" fetch needed a human to
# know roughly which year to ask about -- nothing here is quite as automatic as it looks.
search_query = "Nobel Prize in Literature 2025 laureate awarded"
snippets = wikipedia_search_snippets(search_query)
for snippet in snippets:
    show(snippet)
    print()

Each snippet is clean and unambiguous on its own — Wikipedia keeps one dedicated page per year for each Nobel Prize, and its very first sentence states that year's laureate directly. But notice the search surfaced *three different years* (2025, 2024, 2016): picking out the single most recent one now means comparing across snippets, not just reading one of them. That comparison is exactly what we'll watch the model attempt next.

## Feeding the result back to the model

<img src="images/rag_feeding_vignette.jpg" width="150" alt="Feeding data" style="float: left; margin-right: 15px; margin-bottom: 10px;">



In [ ]:
# Program 4: answering using the fetched text instead of frozen memory

context = "\n".join(snippets)

question = f"""Here are a few short extracts found on Wikipedia:

{context}

Based only on this information, who won the most recent Nobel Prize in Literature? Answer in
one sentence, and say clearly if you are not fully sure."""

result = await Runner.run(research_agent, question)
show(result.final_output)

Run this cell several times, and don't be surprised if the model gets it wrong more often than right. In our own testing (`mistral:latest`, 11 runs), it answered "Bob Dylan" — the 2016 laureate, by far the most famous of the three names — most often, even though the text plainly gives 2025 and 2024 dates too; Han Kang (2024) came up almost as often; László Krasznahorkai (2025, the genuinely correct answer) showed up only occasionally, sometimes with the model second-guessing itself because 2025 felt "too recent" to trust.

This is a different, and in some ways more surprising, failure than the ones coming up with the messier example below: the correct answer really is sitting right there in the text this time, in a snippet just as clean as the other two. But picking out the largest of three years is still a real reasoning step, and a small model doesn't always get a simple comparison right — especially when one of the candidates is far more familiar to it than the others. Grounding a model in real text fixes the *frozen-knowledge* problem from Program 1; it does not automatically fix the model's *reasoning* about what that text says.

Size does help here, though — unlike with Program 7's tool-calling loop below. Running this exact same prompt against bigger, hosted models on the University of Rennes server (Part 1), `ilaas/gpt-oss-120b` and `ilaas/mistral-small-4-119b` both got it right **4 times out of 4**, confidently and consistently. So a bigger model reliably fixes *this* kind of problem — simple reasoning over text that's already in front of it — even though, as we're about to see with the tool-calling agent, a bigger model does *not* reliably fix every kind of problem an LLM can run into.

Notice, too, who made the decision to fetch this text in the first place: **we** did, in our Python code, before ever calling the model a second time. That's still an *algorithmic* workflow (Part 2's distinction) — we call the model, then unconditionally fetch, then call the model again, in a fixed order we wrote by hand.

In [ ]:
# write your code with a RAGAREN model such as ilaas/mistral-small-4-119b 

## A messier example: the current French Minister

<img src="images/french_minister_vignette.jpg" width="150" alt="French minister" style="float: left; margin-right: 15px; margin-bottom: 10px;">

Even with clean source text, Program 4 just showed the model can still get a simple comparison wrong. Let's look at a case where the source data itself is messy and confusing, to see how that affects the model's reasoning.

In [ ]:
# Program 5: the same method, on a messier subject

search_query = "\"ministre de l\'Enseignement supérieur et de la Recherche\" depuis"
snippets = wikipedia_search_snippets(search_query)
for snippet in snippets:
    show(snippet)
    print()

context = "\n".join(snippets)
question = f"""Here are a few short extracts found on Wikipedia:

{context}

Based only on this information, who is the current French Minister of Higher Education and
Research? Answer in one sentence, and say clearly if you are not fully sure."""

result = await Runner.run(research_agent, question)
show(result.final_output)

Honestly imperfect this time: the snippets are short, ambiguous about *which* name is the most recent one, and Wikipedia itself can lag behind reality by days or weeks. A good model will tell you it isn't sure — but don't be surprised if it picks a name anyway. Same method, same code, very different result: the reliability of "fetch, then feed it to the model" depends heavily on how well the underlying source documents the topic.

## What if we ask the model for the sources themselves?

<img src="images/hallucination_vignette.jpg" width="150" alt="Asking for sources" style="float: left; margin-right: 15px; margin-bottom: 10px;">

A tempting fix: instead of *us* picking Wikipedia's search API, why not ask the model which web pages it wants to visit, and then fetch those pages? After all, the model knows what it's trying to find. Let's see what happens if we ask it to give us the URLs.

In [ ]:
# Program 6: checking the model's own suggested sources

import json

question = (
    "What are 2 or 3 web pages (give their exact, real URLs) that would reliably tell me who the "
    "current French Minister of Higher Education and Research is? Respond with ONLY a JSON array "
    "of URL strings, no other text, no markdown formatting."
)
result = await Runner.run(research_agent, question)
show(result.final_output)

print()
try:
    urls = json.loads(result.final_output)
    for url in urls:
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            print(f"{response.status_code}  {url}")
        except requests.RequestException as error:
            print(f"FAILED  {url}  ({error.__class__.__name__})")
except json.JSONDecodeError:
    print("The model didn't return valid JSON this time -- try running the cell again.")

Run this a few times and look at the actual status codes, not just the URLs themselves. In our own testing, this turned up every flavor of failure at once: URLs on domains that don't exist at all (connection failure), real government domains that exist but reject scripted requests (`403`), and specific paths that were simply invented (`404`). Root-level domains the model suggests (`gouvernement.fr`, `assemblee-nationale.fr`) are often genuinely real -- but that alone doesn't get you the specific fact, since a homepage isn't the page that states it, and government sites tend to block simple scripts anyway.

The underlying issue is the same one from Program 1, just wearing a different costume: a model doesn't *know* URLs the way a search index does, it generates plausible-*looking* text -- and a URL is just another string it can pattern-complete confidently and incorrectly, exactly like a name.

## Letting the model decide: our first real agent

<img src="images/agent_steering_vignette.jpg" width="150" alt="Letting the model decide" style="float: left; margin-right: 15px; margin-bottom: 10px;">

We've been reusing the same `research_agent` since Program 1, without ever giving it anything beyond its own frozen knowledge. Let's remove the last piece of hardcoding: instead of *us* deciding to fetch Wikipedia, we give the agent a **tool** — exactly `@function_tool` from Part 2 — and let *it* decide, on its own, whether it needs to call it.

A quick reminder of what `@function_tool` actually does, since it matters for what comes next: it takes an ordinary Python function and wraps it into something the model can be offered as a tool. To do that, it needs to describe the tool to the model in plain text — and it builds that description straight from the function's **docstring** (including the `Args:` section): the docstring isn't just a comment for us, it's *literally* sent to the model as the tool's instructions for when and how to call it. Write a vague or wrong docstring, and the model has nothing better to go on — one more good reason to document your code properly! 🙂

This is really the one place in this notebook where a diagram earns its keep: contrasting the fixed pipeline of Programs 1-6 with this program's model-driven branching.

<br>
<img src="images/workflow_vs_agent_academic.jpg" width="550" alt="Algorithmic Workflow vs Agent Diagram" style="display: block; margin: 15px auto;">
<br>

(I didn't ask for more than this one diagram — the rest of this notebook is really about behavior and reliability, better shown by the actual printed results than by an illustration.)

In [ ]:
# Program 7: giving the model the choice to call the tool itself

from agents import function_tool

@function_tool
def wikipedia_lookup(query: str):
    """Search Wikipedia for a query and return a few short text snippets. Use this whenever
    you need current, factual information you cannot be fully confident about from memory
    alone.

    Args:
        - query: what to search for on Wikipedia.
    """
    print(f">>> tool actually called, with query: {query!r}")  # proof the tool really ran
    return "\n".join(wikipedia_search_snippets(query))

research_agent_with_tool = Agent(
    name="Research Agent",
    instructions="Answer the user's question. If you are not fully confident from memory alone, "
                 "use the wikipedia_lookup tool before answering.",
    model=ollama_model,
    tools=[wikipedia_lookup],
)

question = "Who is the current French Minister of Higher Education and Research?"
result = await Runner.run(research_agent_with_tool, question)
print("Final answer:")
show(result.final_output)

Run this one several times. The debug print is there so you can check, honestly, what actually happened -- and with a small local model like `mistral:latest`, you'll likely see a mix of behaviors: sometimes the tool is genuinely called (the print appears) and the final answer *still* doesn't quite match what it returned; sometimes the model skips the tool call entirely and just answers from memory anyway.

This isn't a bug in our code, and it isn't specific to `mistral:latest` either -- in our own testing, swapping in other local models (`qwen3:8b`, `llama3.2`) just produced *different* failures (one confidently invented a name despite admitting the tool's result didn't support it; the other got confused about whether it had even called the tool). We even tried a much bigger, hosted model (Rennes's ~120-billion-parameter Mistral) -- it called the tool ten times with slightly reworded queries, never satisfied with the answer, and hit a turn limit without ever responding at all. Bigger didn't mean more reliable here, at least not in this exact setup.

This is exactly why, back in Part 2, tools were only ever handed to the Rennes- or OpenAI-hosted agents, never to the Ollama one. It's also a preview of why the next step, **RAG**, is designed the way it is: instead of hoping the model calls the right tool and uses the result faithfully, RAG systems typically retrieve relevant text *before* the model ever runs, and feed it in directly -- closer to what Programs 3-5 did by hand, but done carefully and at scale.

## Key takeaways

<img src="images/takeaways.jpg" width="150" alt="Key takeaways" style="float: left; margin-right: 15px; margin-bottom: 10px;">

* A model's knowledge is frozen at training time: for anything that changes after its training cutoff, it can only guess or hedge -- it cannot know.
* Asking the model itself "how would you find out" is often a good, cheap way to get a sensible retrieval strategy, even if the model cannot execute it directly -- we can read its plan, and build the retrieval ourselves in python (as we did in Program 3).
* Feeding retrieved text back to the model (retrieval-augmented generation, or RAG) is the standard way to fix the frozen-knowledge problem -- but simple RAG only gives the model the right facts, it does not make the model's reasoning about those facts perfect.
* Hallucination is a real problem when asking models for web URLs: they will confidently make up realistic-looking domain names and paths that don't exist -- do not trust a model's own suggestions for external sources.
* Giving a model a tool (agentic workflow) means wrapping a python function into a description (its docstring) that the model reads: the model decides for itself whether and when to output a tool-call request, and our runner intercepts it, runs the python code, and feeds the result back.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `Agent`, `Runner.run`, `OpenAIChatCompletionsModel`, `@function_tool`, and `json.loads`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `requests.get(url, params=, headers=, timeout=)` (`requests`) | a URL, query parameters (dict), HTTP headers (dict), a timeout in seconds | a `Response` object; `.json()` parses its body as JSON, `.status_code` is the HTTP status | Programs 3, 5, 6 |
| `re.sub(pattern, repl, text)` (`re`) | a regex pattern, replacement string, text to clean | the text with every match replaced | Programs 3, 5 |